# RAG with HANA Vector Store (Orchestration SDK)

This notebook demonstrates building a **Retrieval-Augmented Generation (RAG)** pipeline using:
- **SAP HANA Cloud Vector Store** with direct SQL queries for retrieval
- **SAP Cloud SDK for AI Orchestration** (`OrchestrationConfig`) for generation
- **GPT-4o** for answer generation via OrchestrationService

This is the Orchestration SDK approach. For a native Python approach (no orchestration), see the `Native_RAG.ipynb` in the best-practices folder.

## Prerequisites
- SAP HANA Cloud instance with vector engine enabled
- SAP AI Core with deployed embedding and chat models
- Orchestration service deployed on AI Core
- The `SCIENCE_DATA` table must be populated by running `Orchestration_HANA_VectorStore_Embeddings.ipynb` first
- Environment variables configured in a `.env` file (see `.env-example`)

## Setup

### Dependencies
Ensure the libraries mentioned in the `requirements.txt` file are installed.

### Environment Variables
Copy `.env-example` to `.env` and fill in your SAP HANA Cloud and SAP AI Core credentials.

In [ ]:
# Import section
import os
from dotenv import load_dotenv, find_dotenv

from hana_ml import ConnectionContext
from gen_ai_hub.proxy.core.proxy_clients import get_proxy_client
from gen_ai_hub.proxy.native.openai import embeddings
from gen_ai_hub.orchestration.service import OrchestrationService
from gen_ai_hub.orchestration.models.config import OrchestrationConfig
from gen_ai_hub.orchestration.models.llm import LLM
from gen_ai_hub.orchestration.models.message import SystemMessage, UserMessage
from gen_ai_hub.orchestration.models.template import Template, TemplateValue

# Loads configuration from the nearest .env file (searches from CWD upward).
dotenv_path = find_dotenv(usecwd=True)
load_dotenv(dotenv_path=dotenv_path, override=True)

## Retrieval

Use the user's query to retrieve the most semantically similar documents from the vector store. The retrieved context will be used to ground the LLM for answer generation via OrchestrationConfig.

In [ ]:
# Connect to SAP HANA
cc = ConnectionContext(
    address=os.environ.get("HANA_ADDRESS"),
    port=os.environ.get("HANA_PORT"),
    user=os.environ.get("HANA_USER"),
    password=os.environ.get("HANA_PASSWORD"),
    encrypt=True
)

cursor = cc.connection.cursor()

print(cc.hana_version())
print(cc.get_current_schema())

In [ ]:
# Initialize AI Core proxy client
proxy_client = get_proxy_client('gen-ai-hub')

# Embedding model: configurable via environment variable
EMBEDDING_MODEL = os.environ.get("EMBEDDING_MODEL", "text-embedding-3-small")

def get_embedding(query):
    """Create embedding vector for given text."""
    embeds = embeddings.create(
        model_name=EMBEDDING_MODEL,
        input=query
    )
    return embeds.data[0].embedding

In [ ]:
def run_vector_search(query, cursor, table_name, metric="COSINE_SIMILARITY", k=4):
    """Performs vector search on indexed documents."""
    ALLOWED_METRICS = {"COSINE_SIMILARITY", "L2DISTANCE"}
    if metric not in ALLOWED_METRICS:
        raise ValueError(f"metric must be one of {ALLOWED_METRICS}")
    k = int(k)

    query_vector = get_embedding(query)
    if not query_vector:
        raise ValueError("Failed to generate query embedding.")

    sort_order = "DESC" if metric != "L2DISTANCE" else "ASC"
    sql_query = f'''
    SELECT TOP {k} MY_TEXT, MY_METADATA
    FROM "{table_name}"
    ORDER BY {metric}(MY_VECTOR, TO_REAL_VECTOR('{query_vector}')) {sort_order}
    '''
    cursor.execute(sql_query)
    return cursor.fetchall()

In [ ]:
query = "How can you test for the presence of proteins in food?"

# Retrieve top 4 matching docs from vector store
context_records = run_vector_search(query, cursor, "SCIENCE_DATA", "COSINE_SIMILARITY", 4)
# Join the content from retrieved docs (MY_TEXT is column 0)
context = " ".join([c[0] for c in context_records])

print(f"Retrieved {len(context_records)} documents for context.")

## Augment & Generate with OrchestrationConfig

Use the **Orchestration SDK** to define a prompt template with placeholders, configure the LLM, and run the generation pipeline. The retrieved context is injected into the template via `TemplateValue`.

In [ ]:
# Initialize Orchestration Service
orchestration_service = OrchestrationService()

# Configure LLM
llm = LLM(name="gpt-4o", parameters={"temperature": 0.0})

In [ ]:
# Define prompt template with placeholders for context and query
prompt_template = Template(messages=[
    SystemMessage("You are an intelligent assistant. Use the provided context information to answer the user's query accurately and professionally."),
    UserMessage("""Context: {{?context}}

Based on the above context, answer the following query:
{{?query}}

The answer tone has to be very professional in nature.
If you don't know the answer, politely say that you don't know, don't try to make up an answer."""),
])

# Build Orchestration Config
config = OrchestrationConfig(template=prompt_template, llm=llm)

In [ ]:
# Run orchestration with retrieved context
response = orchestration_service.run(
    config=config,
    template_values=[
        TemplateValue(name="context", value=context),
        TemplateValue(name="query", value=query),
    ]
)

print(response.orchestration_result.choices[0].message.content)

## Follow-up Query

Demonstrate reusing the same OrchestrationConfig with a different query.

In [ ]:
# New query
query_2 = "What are the main differences between plant and animal cells?"

# Retrieve context for the new query
context_records_2 = run_vector_search(query_2, cursor, "SCIENCE_DATA", "COSINE_SIMILARITY", 4)
context_2 = " ".join([c[0] for c in context_records_2])

# Run orchestration with the new context and query
response_2 = orchestration_service.run(
    config=config,
    template_values=[
        TemplateValue(name="context", value=context_2),
        TemplateValue(name="query", value=query_2),
    ]
)

print(response_2.orchestration_result.choices[0].message.content)